In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!mkdir -p /content/contract-reviewer/data
!unzip -q -o /content/drive/MyDrive/CUAD_v1.zip -d /content/contract-reviewer/data
!unzip -q -o /content/drive/MyDrive/code.zip -d /content/contract-reviewer
!pip install -q sentence-transformers rank_bm25

%cd /content/contract-reviewer
!mkdir -p notebooks /content/drive/MyDrive/contract-cache /content/drive/MyDrive/rerank-cache
!ln -s /content/drive/MyDrive/contract-cache data/cache
!ln -s /content/drive/MyDrive/rerank-cache data/rerank_cache
%cd notebooks

import torch
print("GPU:", torch.cuda.is_available())
!ls ..

/content/contract-reviewer
/content/contract-reviewer/notebooks
GPU: True
data  eval  notebooks  src


In [3]:
import sys, time
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.rerank import make_rerank_ranker
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])
reranked = make_rerank_ranker(hybrid, top_n=10)

In [4]:
t = time.time()
evaluate_ranker(reranked, contracts, dev_idx[:10], "Governing Law")
print(round(time.time() - t, 1), "seconds for 10 contracts")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

39.8 seconds for 10 contracts


In [5]:
rows = {cl: evaluate_ranker(reranked, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
rr = pd.DataFrame(rows).T
rr.to_csv("../eval/results_hybrid_1_2_rerank10_clause1500_dev.csv")

base = pd.read_csv("../eval/results_hybrid_1_2_clause1500_dev.csv", index_col=0)
cols = ["hit@1", "hit@3", "hit@10", "mrr"]
print(pd.DataFrame({"hybrid_1_2": base[cols].mean(), "+ rerank": rr[cols].mean()}).T.round(3))
print()
print(pd.DataFrame({"hybrid_1_2": base["hit@1"], "+ rerank": rr["hit@1"]}).round(3))

            hit@1  hit@3  hit@10    mrr
hybrid_1_2  0.747  0.921   0.968  0.836
+ rerank    0.711  0.858   0.968  0.799

                                    hybrid_1_2  + rerank
Governing Law                            0.918     0.908
Renewal Term                             0.881     0.924
Notice Period To Terminate Renewal       0.676     0.859
Termination For Convenience              0.712     0.585
Non-Compete                              0.427     0.267
Cap On Liability                         0.816     0.695
Uncapped Liability                       0.802     0.741


In [6]:
!cp ../eval/results_hybrid_1_2_rerank10_clause1500_dev.csv /content/drive/MyDrive/